# Building and integrating a testing framework

This notebook demonstrates how to use the testing framework to evaluate different LLM models.

Source: [LinkedIn Learning - Build A SQL AI Agent for Production](https://www.linkedin.com/learning/build-with-ai-sql-ai-agents-in-production)

Models to Test

In [1]:
import sys
import os
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Add project root to path
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.testing import TestRunner, get_test_cases
from sql_ai_agent.testing.test_cases import get_test_summary

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## View Test Cases

In [2]:
# Get test summary
summary = get_test_summary()

print(f"Total Test Cases: {summary['total_tests']}")
print(f"\nBy Difficulty:")
for diff, count in sorted(summary["by_difficulty"].items()):
    print(f"  {diff.capitalize()}: {count}")

print(f"\nBy Category:")
for cat, count in sorted(summary["by_category"].items()):
    print(f"  {cat}: {count}")


Total Test Cases: 20

By Difficulty:
  Easy: 5
  Hard: 7
  Medium: 8

By Category:
  aggregation: 3
  basic: 2
  comparison: 1
  complex_filtering: 1
  edge_case: 3
  filtering: 2
  growth_analysis: 2
  market_share: 1
  percentage: 1
  ranking: 2
  time_series: 2


In [3]:
# Display all test cases
all_tests = get_test_cases()

test_df = pd.DataFrame(
    [
        {
            "ID": test.id,
            "Difficulty": test.difficulty,
            "Category": test.category,
            "Question": test.question[:80] + "..."
            if len(test.question) > 80
            else test.question,
        }
        for test in all_tests
    ]
)

display(test_df)


,ID,Difficulty,Category,Question
0,1,easy,basic,How many total rows are in the air_traffic table?
1,2,easy,aggregation,What are the top 5 airlines by total passenger count in 2024?
2,3,easy,aggregation,"Show total passengers by terminal, excluding transit passengers"
3,4,easy,filtering,How many international passengers arrived at SFO in 2023?
4,5,easy,basic,List all distinct operating airlines in the database
5,6,medium,comparison,Compare domestic vs international passenger traffic for 2024
6,7,medium,aggregation,What are the top 3 low-fare carriers by passenger count in 2024?
7,8,medium,time_series,Show monthly passenger trends for United Airlines in 2024
8,9,medium,ranking,Which terminal had the highest passenger traffic in January 2024?
9,10,medium,percentage,What percentage of total passengers were on low-fare carriers in 2024?


## Initialize Test Runner

In [4]:
# Initialize the test runner with database connection
runner = TestRunner(
    db_host="postgres",
    db_port=5432,
    db_name="my_db",
    db_user="postgres",
    db_password="password",
    table_name="air_traffic",
)

print("✅ Test runner initialized successfully")


✅ Test runner initialized successfully


## Define Models to Test

In [32]:
# Define models to test
models_to_test = {
    "openai": [
        "gpt-5.5",
        "gpt-5.5-pro",
        "gpt-5.4-nano",
        "gpt-5.4-mini",
        "gpt-4.1",
    ],
    "docker_model_runner": [
        "ai/granite-4.0-h-micro",
        "ai/llama3.2:latest",
        "ai/gemma4:100k"
    ],
}

print("Models to test:")
for provider, models in models_to_test.items():
    print(f"\n{provider.upper()}:")
    for model in models:
        print(f"  - {model}")


Models to test:

OPENAI:
  - gpt-5.5
  - gpt-5.5-pro
  - gpt-5.4-nano
  - gpt-5.4-mini
  - gpt-4.1

DOCKER_MODEL_RUNNER:
  - ai/granite-4.0-h-micro
  - ai/llama3.2:latest
  - ai/gemma4:100k


## Test 1: Run All Tests for All Models
This will run all 20 test cases + 5 debug tests for each model.

In [ ]:
# Run all tests for all models
# WARNING: This may take a while (20 tests + 5 debug tests per model)

print("Starting comprehensive test run...\n")

# Count total models across all providers
total_models = sum(len(models) for models in models_to_test.values())

tests_per_model = 25  # 20 generation + 5 debug

print("This will test:")
print(f"  - {total_models} models across {len(models_to_test)} providers")

# Breakdown per provider (optional but useful)
for provider, models in models_to_test.items():
    print(f"    - {provider}: {len(models)} models")

print(f"  - 20 query generation tests per model")
print(f"  - 5 debug mechanism tests per model")
print(f"  - Total: {total_models * tests_per_model} test executions\n")

# Run tests for all providers
results_df = runner.run_all_tests(
    providers=list(models_to_test.keys()),
    models=models_to_test,
    max_debug_trials=3,
)

print("\n✅ All tests completed!")
print(f"Total results: {len(results_df)} test executions")


Starting comprehensive test run...

This will test:
  - 8 models across 2 providers
    - openai: 5 models
    - docker_model_runner: 3 models
  - 20 query generation tests per model
  - 5 debug mechanism tests per model
  - Total: 200 test executions


################################################################################
# Testing Provider: OPENAI | Model: gpt-5.5
################################################################################

Testing openai/gpt-5.5 - Query Generation (20 tests)

[1/20] Test 1: How many total rows are in the air_traffic table?...
   ✓ PASS (1.54s) - Query executed successfully
[2/20] Test 2: What are the top 5 airlines by total passenger count in 2024...
   ✗ FAIL (2.84s) - Custom validation failed
[3/20] Test 3: Show total passengers by terminal, excluding transit passeng...
   ✓ PASS (2.89s) - Query executed successfully
[4/20] Test 4: How many international passengers arrived at SFO in 2023?...
   ✓ PASS (3.04s) - Query executed success

Traceback (most recent call last):
  File "/workspace/sql_ai_agent/testing/framework.py", line 348, in run_all_tests
    debug_results = self.run_debug_tests(
                    ^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/sql_ai_agent/testing/framework.py", line 268, in run_debug_tests
    debug_results = debug_tester.run_debug_tests(
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/sql_ai_agent/testing/debug_tester.py", line 226, in run_debug_tests
    success, trials, exec_time, final_query, final_error = self.run_single_debug_test(
                                                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspace/sql_ai_agent/testing/debug_tester.py", line 170, in run_single_debug_test
    llm_debug_output = debug_agent(
                       ^^^^^^^^^^^^
  File "/workspace/sql_ai_agent/SqlAgent.py", line 274, in debug_agent
    llm_output = chain.invoke(
                 ^^^^^^^^^^^^^
  File "/opt/sql-ai-agent-dev/lib/python3.11/site-packag

   ✓ PASS (0.69s) - Query executed successfully
[2/20] Test 2: What are the top 5 airlines by total passenger count in 2024...
   ✗ FAIL (0.96s) - Custom validation failed
[3/20] Test 3: Show total passengers by terminal, excluding transit passeng...
   ✓ PASS (0.91s) - Query executed successfully
[4/20] Test 4: How many international passengers arrived at SFO in 2023?...
   ✓ PASS (1.24s) - Query executed successfully
[5/20] Test 5: List all distinct operating airlines in the database...
   ✓ PASS (0.86s) - Query executed successfully
[6/20] Test 6: Compare domestic vs international passenger traffic for 2024...
   ✗ FAIL (10.66s) - WITH query name "classified" specified more than once
LINE 1: ... NOT traffic_type IS NULL GROUP BY traffic_type), classified...
                                                             ^
[7/20] Test 7: What are the top 3 low-fare carriers by passenger count in 2...
   ✗ FAIL (1.16s) - Too few rows: 0 < 1
[8/20] Test 8: Show monthly passenger trends fo

## Save Results to CSV

In [ ]:
# Generate timestamp for file naming
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save detailed results
results_file = f"test_results_{timestamp}.csv"
results_df.to_csv(results_file, index=False)
print(f"✅ Detailed results saved to: {results_file}")

# Generate and save summary
summary_df = runner.generate_summary_report(results_df)
summary_file = f"test_summary_{timestamp}.csv"
summary_df.to_csv(summary_file, index=False)
print(f"✅ Summary report saved to: {summary_file}")


✅ Detailed results saved to: test_results_20260503_141848.csv
✅ Summary report saved to: test_summary_20260503_141848.csv


## View Summary Results

In [15]:
# Display summary report
print("\n" + "=" * 100)
print("TEST SUMMARY")
print("=" * 100 + "\n")

display(summary_df)



TEST SUMMARY



,provider,model,test_type,total_tests,successful,failed,success_rate,avg_execution_time,avg_trials_to_fix
0,docker_model_runner,ai/gemma4:100k,debug_mechanism,5,5,0,100.00%,1.483s,1.20
1,docker_model_runner,ai/gemma4:100k,query_generation,20,13,7,65.00%,2.627s,NaN
2,docker_model_runner,ai/granite-4.0-h-micro,debug_mechanism,5,4,1,80.00%,1.678s,1.00
3,docker_model_runner,ai/granite-4.0-h-micro,query_generation,20,14,6,70.00%,2.882s,NaN
4,docker_model_runner,ai/llama3.2:latest,debug_mechanism,5,4,1,80.00%,0.967s,1.00
5,docker_model_runner,ai/llama3.2:latest,query_generation,20,8,12,40.00%,1.760s,NaN
6,docker_model_runner,ai/qwen3.6_27b:128k,query_generation,20,0,20,0.00%,3.071s,NaN
7,docker_model_runner,ai/qwen3._9b:128k,query_generation,20,0,20,0.00%,0.002s,NaN


## Analyze Results by Test Type

In [16]:
# Query Generation Results
query_results = results_df[
    results_df["test_type"] == "query_generation"
].copy()

print("\n" + "=" * 80)
print("QUERY GENERATION RESULTS")
print("=" * 80 + "\n")

query_summary = (
    query_results.groupby("model")
    .agg({"success": ["count", "sum", "mean"], "execution_time": "mean"})
    .round(3)
)

query_summary.columns = [
    "Total Tests",
    "Successful",
    "Success Rate",
    "Avg Time (s)",
]
query_summary["Success Rate"] = (query_summary["Success Rate"] * 100).round(
    2
).astype(str) + "%"

display(query_summary.sort_values("Successful", ascending=False))



QUERY GENERATION RESULTS



,Total Tests,Successful,Success Rate,Avg Time (s)
model,,,,
ai/granite-4.0-h-micro,20,14,70.0%,2.882
ai/gemma4:100k,20,13,65.0%,2.627
ai/llama3.2:latest,20,8,40.0%,1.760
ai/qwen3.6_27b:128k,20,0,0.0%,3.071
ai/qwen3._9b:128k,20,0,0.0%,0.002


## Analyze Results by Difficulty

In [17]:
# Success rate by difficulty level
if "difficulty" in query_results.columns:
    print("\n" + "=" * 80)
    print("SUCCESS RATE BY DIFFICULTY")
    print("=" * 80 + "\n")

    difficulty_pivot = (
        pd.crosstab(
            query_results["model"],
            query_results["difficulty"],
            query_results["success"],
            aggfunc="mean",
        ).round(3)
        * 100
    )

    # Reorder columns: easy, medium, hard
    column_order = [
        col
        for col in ["easy", "medium", "hard"]
        if col in difficulty_pivot.columns
    ]
    difficulty_pivot = difficulty_pivot[column_order]

    display(difficulty_pivot.style.format("{:.2f}%"))



SUCCESS RATE BY DIFFICULTY



difficulty,easy,medium,hard
model,,,
ai/gemma4:100k,80.00%,50.00%,71.40%
ai/granite-4.0-h-micro,80.00%,50.00%,85.70%
ai/llama3.2:latest,60.00%,25.00%,42.90%
ai/qwen3.6_27b:128k,0.00%,0.00%,0.00%
ai/qwen3._9b:128k,0.00%,0.00%,0.00%


In [ ]:
# Debug Mechanism Results
debug_results = results_df[results_df["test_type"] == "debug_mechanism"].copy()

if not debug_results.empty:
    print("\n" + "=" * 80)
    print("DEBUG MECHANISM RESULTS")
    print("=" * 80 + "\n")

    debug_summary = (
        debug_results.groupby("model")
        .agg(
            {
                "success": ["count", "sum", "mean"],
                "trials_needed": "mean",
                "execution_time": "mean",
            }
        )
        .round(3)
    )

    debug_summary.columns = [
        "Total Tests",
        "Fixed",
        "Fix Rate",
        "Avg Trials",
        "Avg Time (s)",
    ]
    debug_summary["Fix Rate"] = (debug_summary["Fix Rate"] * 100).round(
        2
    ).astype(str) + "%"

    display(debug_summary.sort_values("Fixed", ascending=False))



DEBUG MECHANISM RESULTS



,Total Tests,Fixed,Fix Rate,Avg Trials,Avg Time (s)
model,,,,,
ai/gemma4:100k,5,5,100.0%,1.2,1.483
ai/granite-4.0-h-micro,5,4,80.0%,1.4,1.678
ai/llama3.2:latest,5,4,80.0%,1.4,0.967


## Visualizations

In [20]:
# Success Rate Comparison
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Prepare data
query_success = (
    query_results.groupby("model")["success"]
    .mean()
    .sort_values(ascending=False)
    * 100
)
query_success_df = query_success.reset_index()
query_success_df.columns = ["model", "success_rate"]

# Create subplots with 1 row and 2 columns
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Query Generation Success Rate by Model",
        "Debug Fix Rate by Model",
    ),
)

# Query Generation Success Rate (left subplot)
fig.add_trace(
    go.Bar(
        x=query_success_df["model"],
        y=query_success_df["success_rate"],
        name="Query Success",
        marker_color="steelblue",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Debug Fix Rate (right subplot)
if not debug_results.empty:
    debug_success = (
        debug_results.groupby("model")["success"]
        .mean()
        .sort_values(ascending=False)
        * 100
    )
    debug_success_df = debug_success.reset_index()
    debug_success_df.columns = ["model", "fix_rate"]

    fig.add_trace(
        go.Bar(
            x=debug_success_df["model"],
            y=debug_success_df["fix_rate"],
            name="Debug Fix",
            marker_color="coral",
            showlegend=False,
        ),
        row=1,
        col=2,
    )

# Update layout
fig.update_xaxes(title_text="Model", row=1, col=1)
fig.update_xaxes(title_text="Model", row=1, col=2)
fig.update_yaxes(title_text="Success Rate (%)", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="Fix Rate (%)", range=[0, 100], row=1, col=2)

fig.update_layout(
    height=500, width=1200, showlegend=False, template="plotly_white"
)

fig.write_html(f"success_rates_{timestamp}.html")
fig.show()


In [21]:
# Execution Time Comparison
import plotly.graph_objects as go

avg_times = (
    query_results.groupby("model")["execution_time"].mean().sort_values()
)
avg_times_df = avg_times.reset_index()
avg_times_df.columns = ["model", "avg_time"]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=avg_times_df["avg_time"],
        y=avg_times_df["model"],
        orientation="h",
        marker_color="seagreen",
        text=avg_times_df["avg_time"].round(3),
        textposition="auto",
    )
)

fig.update_layout(
    title="Average Query Execution Time by Model",
    xaxis_title="Average Execution Time (seconds)",
    yaxis_title="Model",
    height=500,
    width=900,
    template="plotly_white",
    showlegend=False,
)

fig.write_html(f"execution_times_{timestamp}.html")
fig.show()


In [22]:
# Success Rate by Difficulty (Heatmap)
if "difficulty" in query_results.columns:
    import plotly.graph_objects as go

    difficulty_pivot = (
        pd.crosstab(
            query_results["model"],
            query_results["difficulty"],
            query_results["success"],
            aggfunc="mean",
        )
        * 100
    )

    # Reorder columns: easy, medium, hard
    column_order = [
        col
        for col in ["easy", "medium", "hard"]
        if col in difficulty_pivot.columns
    ]
    difficulty_pivot = difficulty_pivot[column_order]

    # Create heatmap
    fig = go.Figure(
        data=go.Heatmap(
            z=difficulty_pivot.values,
            x=difficulty_pivot.columns,
            y=difficulty_pivot.index,
            colorscale="RdYlGn",
            zmin=0,
            zmax=100,
            text=difficulty_pivot.values.round(1),
            texttemplate="%{text:.1f}",
            textfont={"size": 12},
            colorbar=dict(title="Success Rate (%)"),
        )
    )

    fig.update_layout(
        title="Success Rate by Model and Difficulty",
        xaxis_title="Difficulty Level",
        yaxis_title="Model",
        height=600,
        width=800,
        template="plotly_white",
    )

    fig.write_html(f"difficulty_heatmap_{timestamp}.html")
    fig.show()


## Detailed Failure Analysis

In [23]:
# Show failed query generation tests
failed_queries = query_results[query_results["success"] == False].copy()

if not failed_queries.empty:
    print("\n" + "=" * 80)
    print(f"FAILED QUERY TESTS ({len(failed_queries)} failures)")
    print("=" * 80 + "\n")

    failure_summary = (
        failed_queries.groupby(["model", "test_id"])
        .size()
        .reset_index(name="count")
    )

    print("Failures by Model and Test ID:")
    display(
        failure_summary.pivot(index="test_id", columns="model", values="count")
        .fillna(0)
        .astype(int)
    )

    # Show details of failed tests
    print("\nFailed Test Details:")
    failed_details = failed_queries[
        [
            "model",
            "test_id",
            "question",
            "difficulty",
            "category",
            "error_message",
            "validation_message",
        ]
    ].head(10)
    display(failed_details)
else:
    print("\n🎉 All query generation tests passed!")



FAILED QUERY TESTS (65 failures)

Failures by Model and Test ID:


model,ai/gemma4:100k,ai/granite-4.0-h-micro,ai/llama3.2:latest,ai/qwen3.6_27b:128k,ai/qwen3._9b:128k
test_id,,,,,
1,0,0,0,1,1
2,1,1,1,1,1
3,0,0,0,1,1
4,0,0,1,1,1
5,0,0,0,1,1
6,1,1,1,1,1
7,1,1,1,1,1
8,0,0,1,1,1
9,0,0,0,1,1



Failed Test Details:


,model,test_id,question,difficulty,category,error_message,validation_message
1,ai/granite-4.0-h-micro,2,What are the top 5 airlines by total passenger count in 2024?,easy,aggregation,None,Custom validation failed
5,ai/granite-4.0-h-micro,6,Compare domestic vs international passenger traffic for 2024,medium,comparison,"column ""GEO_Region"" does not exist\nLINE 1: ...ta_6x4kaj2wwraxvjd4dumlp5p4xy AS SELECT CASE WHEN...","column ""GEO_Region"" does not exist\nLINE 1: ...ta_6x4kaj2wwraxvjd4dumlp5p4xy AS SELECT CASE WHEN..."
6,ai/granite-4.0-h-micro,7,What are the top 3 low-fare carriers by passenger count in 2024?,medium,aggregation,None,Too few rows: 0 < 1
11,ai/granite-4.0-h-micro,12,Which GEO Region had the most international passengers in 2023?,medium,ranking,"column ""geo_region"" does not exist\nLINE 1: ...res_metadata_gamq53kwtzhe5ibd3jkdz5g4aq AS SELECT...","column ""geo_region"" does not exist\nLINE 1: ...res_metadata_gamq53kwtzhe5ibd3jkdz5g4aq AS SELECT..."
13,ai/granite-4.0-h-micro,14,Which 5 airlines had the highest year-over-year growth from 2023 to 2024?,hard,growth_analysis,"column reference ""Operating Airline"" is ambiguous\nLINE 1: ...res_metadata_txl6zkliz5hppfib5swro...","column reference ""Operating Airline"" is ambiguous\nLINE 1: ...res_metadata_txl6zkliz5hppfib5swro..."
17,ai/granite-4.0-h-micro,18,Show total enplaned passengers only (departures) for each month in 2024,medium,edge_case,None,Too few rows: 0 < 1
26,ai/llama3.2:latest,2,What are the top 5 airlines by total passenger count in 2024?,easy,aggregation,"Validation Error: Unable to parse query: Expecting ). Line 1, Col: 45.\n SELECT Operating Airli...","Validation Error: Unable to parse query: Expecting ). Line 1, Col: 45.\n SELECT Operating Airli..."
28,ai/llama3.2:latest,4,How many international passengers arrived at SFO in 2023?,easy,filtering,"Validation Error: Unable to parse query: Expecting ). Line 1, Col: 31.\n SELECT COUNT(T1.Passen...","Validation Error: Unable to parse query: Expecting ). Line 1, Col: 31.\n SELECT COUNT(T1.Passen..."
30,ai/llama3.2:latest,6,Compare domestic vs international passenger traffic for 2024,medium,comparison,"Validation Error: Unable to parse query: Expected END after CASE. Line 3, Col: 33.\n SELECT \n ...","Validation Error: Unable to parse query: Expected END after CASE. Line 3, Col: 33.\n SELECT \n ..."
31,ai/llama3.2:latest,7,What are the top 3 low-fare carriers by passenger count in 2024?,medium,aggregation,"column ""year"" does not exist\nLINE 1: ...NT(*) AS ""Passenger Count"" FROM air_traffic WHERE Year ...","column ""year"" does not exist\nLINE 1: ...NT(*) AS ""Passenger Count"" FROM air_traffic WHERE Year ..."


In [24]:
# Show failed debug tests
if not debug_results.empty:
    failed_debug = debug_results[debug_results["success"] == False].copy()

    if not failed_debug.empty:
        print("\n" + "=" * 80)
        print(f"FAILED DEBUG TESTS ({len(failed_debug)} failures)")
        print("=" * 80 + "\n")

        failed_debug_details = failed_debug[
            [
                "model",
                "test_id",
                "description",
                "error_type",
                "trials_needed",
                "final_error",
            ]
        ]
        display(failed_debug_details)
    else:
        print("\n🎉 All debug tests passed!")



FAILED DEBUG TESTS (2 failures)



,model,test_id,description,error_type,trials_needed,final_error
20,ai/granite-4.0-h-micro,1,Missing quotes around column names with spaces,syntax_error_quotes,3.0,"Validation Error: Unable to parse query: Expecting ). Line 1, Col: 45.\n SELECT Operating Airli..."
45,ai/llama3.2:latest,1,Missing quotes around column names with spaces,syntax_error_quotes,3.0,"Validation Error: Unable to parse query: Expecting ). Line 1, Col: 45.\n SELECT Operating Airli..."


## Model Comparison Summary

In [25]:
# Create comprehensive comparison table
comparison_data = []

for model in models_to_test["openai"]:
    model_query = query_results[query_results["model"] == model]
    model_debug = (
        debug_results[debug_results["model"] == model]
        if not debug_results.empty
        else pd.DataFrame()
    )

    row = {
        "Model": model,
        "Query Tests": len(model_query),
        "Query Success": model_query["success"].sum(),
        "Query Success %": f"{(model_query['success'].mean() * 100):.2f}%"
        if len(model_query) > 0
        else "N/A",
        "Avg Query Time (s)": f"{model_query['execution_time'].mean():.3f}"
        if len(model_query) > 0
        else "N/A",
        "Debug Tests": len(model_debug),
        "Debug Fixed": model_debug["success"].sum()
        if len(model_debug) > 0
        else 0,
        "Debug Fix %": f"{(model_debug['success'].mean() * 100):.2f}%"
        if len(model_debug) > 0
        else "N/A",
        "Avg Trials": f"{model_debug[model_debug['success'] == True]['trials_needed'].mean():.2f}"
        if len(model_debug[model_debug["success"] == True]) > 0
        else "N/A",
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "=" * 120)
print("COMPREHENSIVE MODEL COMPARISON")
print("=" * 120 + "\n")

display(comparison_df)

# Save comparison
comparison_df.to_csv(f"model_comparison_{timestamp}.csv", index=False)
print(f"\n✅ Model comparison saved to: model_comparison_{timestamp}.csv")


KeyError: 'openai'

In [26]:
# Create comprehensive comparison table
comparison_data = []

for provider, models in models_to_test.items():
    for model in models:
        model_query = (
            query_results[
                (query_results["model"] == model)
                & (query_results["provider"] == provider)
            ]
            if "provider" in query_results.columns
            else query_results[query_results["model"] == model]
        )

        model_debug = pd.DataFrame()
        if not debug_results.empty:
            model_debug = (
                debug_results[
                    (debug_results["model"] == model)
                    & (debug_results["provider"] == provider)
                ]
                if "provider" in debug_results.columns
                else debug_results[debug_results["model"] == model]
            )

        # Pre-calc for cleaner logic
        query_len = len(model_query)
        debug_len = len(model_debug)

        query_success_mean = (
            model_query["success"].mean() if query_len > 0 else None
        )
        debug_success_mean = (
            model_debug["success"].mean() if debug_len > 0 else None
        )

        successful_debug = (
            model_debug[model_debug["success"] == True]
            if debug_len > 0
            else pd.DataFrame()
        )

        row = {
            "Provider": provider,
            "Model": model,
            "Query Tests": query_len,
            "Query Success": model_query["success"].sum()
            if query_len > 0
            else 0,
            "Query Success %": f"{(query_success_mean * 100):.2f}%"
            if query_success_mean is not None
            else "N/A",
            "Avg Query Time (s)": f"{model_query['execution_time'].mean():.3f}"
            if query_len > 0
            else "N/A",
            "Debug Tests": debug_len,
            "Debug Fixed": model_debug["success"].sum()
            if debug_len > 0
            else 0,
            "Debug Fix %": f"{(debug_success_mean * 100):.2f}%"
            if debug_success_mean is not None
            else "N/A",
            "Avg Trials": f"{successful_debug['trials_needed'].mean():.2f}"
            if len(successful_debug) > 0
            else "N/A",
        }

        comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "=" * 120)
print("COMPREHENSIVE MODEL COMPARISON")
print("=" * 120 + "\n")

display(comparison_df)

# Save comparison
comparison_df.to_csv(f"model_comparison_{timestamp}.csv", index=False)
print(f"\n✅ Model comparison saved to: model_comparison_{timestamp}.csv")



COMPREHENSIVE MODEL COMPARISON



,Provider,Model,Query Tests,Query Success,Query Success %,Avg Query Time (s),Debug Tests,Debug Fixed,Debug Fix %,Avg Trials
0,docker_model_runner,ai/granite-4.0-h-micro,20,14,70.00%,2.882,5,4,80.00%,1.00
1,docker_model_runner,ai/llama3.2:latest,20,8,40.00%,1.760,5,4,80.00%,1.00
2,docker_model_runner,ai/gemma4:100k,20,13,65.00%,2.627,5,5,100.00%,1.20
3,docker_model_runner,ai/qwen3._9b:128k,20,0,0.00%,0.002,0,0,N/A,N/A
4,docker_model_runner,ai/qwen3.6_27b:128k,20,0,0.00%,3.071,0,0,N/A,N/A



✅ Model comparison saved to: model_comparison_20260503_141848.csv


## Key Findings Summary

In [27]:
print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80 + "\n")

# Best performing model for query generation
best_query_model = query_results.groupby("model")["success"].mean().idxmax()
best_query_rate = query_results.groupby("model")["success"].mean().max() * 100
print(
    f"🏆 Best Query Generation Model: {best_query_model} ({best_query_rate:.2f}% success rate)"
)

# Fastest model
fastest_model = (
    query_results.groupby("model")["execution_time"].mean().idxmin()
)
fastest_time = query_results.groupby("model")["execution_time"].mean().min()
print(
    f"⚡ Fastest Model: {fastest_model} ({fastest_time:.3f}s avg execution time)"
)

# Best debug model
if not debug_results.empty:
    best_debug_model = (
        debug_results.groupby("model")["success"].mean().idxmax()
    )
    best_debug_rate = (
        debug_results.groupby("model")["success"].mean().max() * 100
    )
    print(
        f"🔧 Best Debug Model: {best_debug_model} ({best_debug_rate:.2f}% fix rate)"
    )

# Overall statistics
total_tests = len(results_df)
total_success = results_df["success"].sum()
overall_rate = (total_success / total_tests * 100) if total_tests > 0 else 0
print(f"\n📊 Overall Statistics:")
print(f"   Total Tests: {total_tests}")
print(f"   Total Successful: {total_success}")
print(f"   Overall Success Rate: {overall_rate:.2f}%")

print("\n" + "=" * 80)



KEY FINDINGS

🏆 Best Query Generation Model: ai/granite-4.0-h-micro (70.00% success rate)
⚡ Fastest Model: ai/qwen3._9b:128k (0.002s avg execution time)
🔧 Best Debug Model: ai/gemma4:100k (100.00% fix rate)

📊 Overall Statistics:
   Total Tests: 115
   Total Successful: 48
   Overall Success Rate: 41.74%

